In [ ]:
from datasets import load_from_disk
from collections import Counter
import re
import math
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from typing import Iterable, List, Optional, Tuple, Dict
import matplotlib.pyplot as plt

In [ ]:
merged = load_from_disk("data/fact_annot_combined")
fact_annot_ds = dict(merged)  # {config_name: Dataset}
len(fact_annot_ds), list(fact_annot_ds)[:3]

In [ ]:
# --- Constants ---
LABEL_RANK: dict[str, int] = {
    "No Issues": 0,
    "N/A - Not Applicable": 0,
    "Not Sure": 1,
    "Minor Issue(s)": 2,
    "Clear Issue(s)": 3,
}
ERROR_LABELS = {"Minor Issue(s)", "Clear Issue(s)"}  # strict errors
ALL_LABELS = ["No Issues", "N/A - Not Applicable", "Not Sure", "Minor Issue(s)", "Clear Issue(s)"]

# --- Matplotlib defaults (clean look) ---
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.alpha": 0.4,
})

In [ ]:
def cfg_to_lang(cfg: str) -> str:
    m = re.search(r"smoldoc__([a-z]{2})", cfg)
    return m.group(1) if m else "xx"

def majority_or_worst(labels: Iterable[Optional[str]]) -> Tuple[Optional[str], Optional[int]]:
    """Majority vote over non-empty labels; tie -> worst by LABEL_RANK."""
    vals = [l for l in labels if l not in (None, "")]
    if not vals:
        return None, None
    counts = Counter(vals).most_common()
    chosen = counts[0][0] if len(counts) == 1 or (len(counts) > 1 and counts[0][1] > counts[1][1]) \
             else max(vals, key=lambda l: LABEL_RANK[l])
    return chosen, LABEL_RANK[chosen]

def pairwise_counts(a: List[Optional[str]], b: List[Optional[str]]):
    """Return (confusion Counter, sorted labels) for Cohen's kappa."""
    conf = Counter()
    labels = set()
    for x, y in zip(a, b):
        if x in (None, "") or y in (None, ""):
            continue
        conf[(x, y)] += 1
        labels.add(x); labels.add(y)
    return conf, sorted(labels)

def cohen_kappa_from_confusion(conf: Counter, labels: List[str]) -> Optional[float]:
    """Compute Cohen’s κ from a confusion counter; None if degenerate."""
    if not labels:
        return None
    n = sum(conf.values())
    if n == 0:
        return None
    po = sum(conf.get((l, l), 0) for l in labels) / n
    row = {l: sum(conf.get((l, j), 0) for j in labels) for l in labels}
    col = {l: sum(conf.get((i, l), 0) for i in labels) for l in labels}
    pe = sum((row[l] / n) * (col[l] / n) for l in labels)
    if pe == 1.0:
        return None
    return (po - pe) / (1 - pe)

In [ ]:
def analyze_configs_strict(datasets_dict: Dict[str, "Dataset"]):
    per_cfg = {}
    global_label_counts = Counter()
    global_error_label_counts = Counter()
    global_total = 0
    global_error_count = 0

    for cfg, dsd in datasets_dict.items():
        cols = set(dsd.column_names)
        required = {"annotator_1_label", "annotator_2_label", "annotator_3_label"}
        if not required.issubset(cols):
            continue  # skip configs not yet augmented

        a1, a2, a3 = dsd["annotator_1_label"], dsd["annotator_2_label"], dsd["annotator_3_label"]
        total = len(a1)

        label_counts = Counter()
        error_label_counts = Counter()
        error_count = 0

        for i in range(total):
            chosen, _ = majority_or_worst([a1[i], a2[i], a3[i]])
            if chosen is None:
                continue
            label_counts[chosen] += 1
            if chosen in ERROR_LABELS:
                error_count += 1
                error_label_counts[chosen] += 1

        per_cfg[cfg] = {
            "total": total,
            "error_count": error_count,
            "error_pct": (error_count / total) * 100 if total else 0.0,
            "label_counts": dict(label_counts),
            "label_pct": {k: (v / total) * 100 for k, v in label_counts.items()} if total else {},
            "error_label_pct_within_errors": (
                {k: (v / error_count) * 100 for k, v in error_label_counts.items()} if error_count else {}
            ),
        }

        global_label_counts.update(label_counts)
        global_error_label_counts.update(error_label_counts)
        global_total += total
        global_error_count += error_count

    global_summary = {
        "total": global_total,
        "error_count": global_error_count,
        "error_pct": (global_error_count / global_total) * 100 if global_total else 0.0,
        "label_counts": dict(global_label_counts),
        "label_pct": {k: (v / global_total) * 100 for k, v in global_label_counts.items()} if global_total else {},
        "error_label_pct_of_all": (
            {k: (v / global_total) * 100 for k, v in global_error_label_counts.items()} if global_total else {}
        ),
        "error_label_pct_within_errors": (
            {k: (v / global_error_count) * 100 for k, v in global_error_label_counts.items()} if global_error_count else {}
        ),
    }
    return per_cfg, global_summary

per_cfg, global_summary = analyze_configs_strict(fact_annot_ds)

# Handy tables you can display():
df_err = (
    pd.DataFrame([
        {"config": cfg,
         "language": cfg_to_lang(cfg),
         "error_pct": st["error_pct"],
         "error_count": st["error_count"],
         "total": st["total"]}
        for cfg, st in per_cfg.items()
    ]).sort_values("error_pct", ascending=False)
)

glob_labels = pd.DataFrame(
    sorted(global_summary["label_pct"].items(), key=lambda x: x[1], reverse=True),
    columns=["label", "pct"]
)

err_mix = pd.DataFrame(
    sorted(global_summary["error_label_pct_within_errors"].items(), key=lambda x: x[1], reverse=True),
    columns=["label", "pct_within_errors"]
)

# Stacked % table [config x label]
df_stack = (
    pd.DataFrame([
        {"config": cfg, **{lbl: per_cfg[cfg]["label_pct"].get(lbl, 0.0) for lbl in ALL_LABELS}}
        for cfg in per_cfg
    ]).set_index("config")
)

In [ ]:
def compute_iaa(datasets_dict: Dict[str, "Dataset"]) -> pd.DataFrame:
    rows = []
    for cfg, dsd in datasets_dict.items():
        cols = set(dsd.column_names)
        if not {"annotator_1_label", "annotator_2_label", "annotator_3_label"}.issubset(cols):
            continue
        a1, a2, a3 = dsd["annotator_1_label"], dsd["annotator_2_label"], dsd["annotator_3_label"]
        total = len(a1)
        disag = (sum(1 for x, y, z in zip(a1, a2, a3) if not (x == y == z)) / total * 100) if total else 0.0

        conf12, labs12 = pairwise_counts(a1, a2)
        conf13, labs13 = pairwise_counts(a1, a3)
        conf23, labs23 = pairwise_counts(a2, a3)
        k12 = cohen_kappa_from_confusion(conf12, labs12)
        k13 = cohen_kappa_from_confusion(conf13, labs13)
        k23 = cohen_kappa_from_confusion(conf23, labs23)

        rows.append({
            "config": cfg,
            "language": cfg_to_lang(cfg),
            "disagreement_pct": disag,
            "kappa_12": k12, "kappa_13": k13, "kappa_23": k23,
            "total": total
        })
    df = pd.DataFrame(rows)
    df["kappa_mean"] = df[["kappa_12","kappa_13","kappa_23"]].mean(axis=1, skipna=True)
    return df.sort_values("disagreement_pct", ascending=False)

df_iaa = compute_iaa(fact_annot_ds)

In [ ]:
def compute_severity(datasets_dict: Dict[str, "Dataset"]) -> pd.DataFrame:
    rows = []
    for cfg, dsd in datasets_dict.items():
        cols = set(dsd.column_names)
        if not {"annotator_1_label", "annotator_2_label", "annotator_3_label"}.issubset(cols):
            continue
        a1, a2, a3 = dsd["annotator_1_label"], dsd["annotator_2_label"], dsd["annotator_3_label"]
        total = len(a1)

        worst_ranks = []
        minor, clear = 0, 0
        for x, y, z in zip(a1, a2, a3):
            vals = [v for v in (x, y, z) if v not in (None, "")]
            if vals:
                worst_ranks.append(max(LABEL_RANK[v] for v in vals))
                maj, _ = majority_or_worst(vals)
                if maj == "Minor Issue(s)":
                    minor += 1
                elif maj == "Clear Issue(s)":
                    clear += 1
            else:
                worst_ranks.append(0)

        mean_worst = float(np.mean(worst_ranks)) if total else 0.0
        rows.append({
            "config": cfg,
            "language": cfg_to_lang(cfg),
            "mean_worst_rank": mean_worst,
            "minor_pct": (minor/total)*100 if total else 0.0,
            "clear_pct": (clear/total)*100 if total else 0.0,
            "total": total
        })
    return pd.DataFrame(rows).sort_values("mean_worst_rank", ascending=False)

df_sev = compute_severity(fact_annot_ds)

In [ ]:
def barplot_with_labels(x, y, *, title: str, xlabel: str, ylabel: str,
                        xtick_rotation: int = 45, value_fmt: str = "{:.1f}",
                        annotate_rotation: int = 0, figsize=(16, 7),
                        bottom_margin: float = 0.25, xtick_fontsize: int = 8):
    plt.figure(figsize=figsize)
    bars = plt.bar(x, y)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=xtick_rotation, ha="right", fontsize=xtick_fontsize)
    plt.subplots_adjust(bottom=bottom_margin)
    for b in bars:
        h = b.get_height()
        if not (isinstance(h, float) and math.isnan(h)):
            plt.text(b.get_x() + b.get_width()/2, h + max(0.01*h, 0.3),
                     value_fmt.format(h), ha="center", va="bottom",
                     rotation=annotate_rotation, fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Stacked bar chart, no loops building 'bottom' by hand (use cumulative sum).
fig, ax = plt.subplots(figsize=(22, 9))
cum = np.zeros(len(df_stack))
xpos = np.arange(len(df_stack))
for lbl in ALL_LABELS:
    vals = df_stack[lbl].values
    ax.bar(df_stack.index, vals, bottom=cum, label=lbl)
    cum = cum + vals

ax.set_title("Per-Config Label Breakdown (Stacked %)")
ax.set_xlabel("SmolDoc Config")
ax.set_ylabel("Percentage of Examples (%)")
plt.xticks(rotation=90, ha="right", fontsize=8)
plt.legend(title="Label", ncol=2)
plt.subplots_adjust(bottom=0.35)
plt.tight_layout()
plt.show()

In [ ]:
# Error rate per config
barplot_with_labels(
    x=df_err["config"], y=df_err["error_pct"],
    title="Factuality Error Rate per SmolDoc Config (Majority of Annotators)",
    xlabel="SmolDoc Config", ylabel="Error Rate (%) — strict (rank ≥ 2)",
    xtick_rotation=45, annotate_rotation=45, figsize=(20, 8), bottom_margin=0.30
)

# Global label distribution (of all examples)
barplot_with_labels(
    x=glob_labels["label"], y=glob_labels["pct"],
    title="Global Label Distribution (All Languages)",
    xlabel="Label", ylabel="Share of All Examples (%)",
    xtick_rotation=45, value_fmt="{:.1f}%"
)

# Error-type mix (within errors)
barplot_with_labels(
    x=err_mix["label"], y=err_mix["pct_within_errors"],
    title="Global Error-Type Mix (Within Errors Only)",
    xlabel="Error Label", ylabel="Share Within Errors (%)",
    xtick_rotation=45, annotate_rotation=90, value_fmt="{:.1f}%", figsize=(10, 6)
)

# Inter-annotator: disagreement %
barplot_with_labels(
    x=df_iaa["config"], y=df_iaa["disagreement_pct"],
    title="Inter-Annotator Disagreement by Config (Not all three equal)",
    xlabel="SmolDoc Config", ylabel="Disagreement (%)",
    xtick_rotation=45, annotate_rotation=40, figsize=(20, 7), bottom_margin=0.30
)

# Inter-annotator: mean Cohen’s κ
barplot_with_labels(
    x=df_iaa["config"], y=df_iaa["kappa_mean"],
    title="Inter-Annotator Agreement (Cohen’s κ, mean of pairs)",
    xlabel="SmolDoc Config", ylabel="Cohen’s κ",
    xtick_rotation=45, annotate_rotation=40, value_fmt="{:.2f}", figsize=(20, 7), bottom_margin=0.30
)

# Severity: mean worst rank
df_sev_cfg = df_sev.sort_values("mean_worst_rank", ascending=False)
barplot_with_labels(
    x=df_sev_cfg["config"], y=df_sev_cfg["mean_worst_rank"],
    title="Error Severity per Config (Mean Worst Label Rank)",
    xlabel="SmolDoc Config", ylabel="Mean Worst Label Rank (0..3)",
    xtick_rotation=45, annotate_rotation=45, figsize=(20, 7), bottom_margin=0.30
)